# inplace-param-update composite — cx16: no-grad eval forward then in-place leaf update on the train batch

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `inference-mode-step`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "inplace-param-update"
DD_ATOM_IDS = ["inference-mode-step", "inplace-param-update"]
DD_SUBTOPICS = ["PyTorch: Inference mode step", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Mid-training validation followed by an in-place parameter update tests a subtle invariant: `p.data.add_(...)` (or any `_`-suffixed leaf write) only works when autograd grad-tracking is OFF — otherwise PyTorch raises:
`RuntimeError: a leaf Variable that requires grad is being used in an in-place operation`.

**Atom A — inference-mode-step.** The eval forward is wrapped in `t.no_grad()` — the standard pattern. `t.inference_mode()` is the even-stricter variant.

**Atom B — inplace-param-update.** The optimizer step is `p.data.add_(g, alpha=-lr)` (or equivalent in-place call on `p.data`). Going through `.data` is what makes this legal even on a leaf tensor with `requires_grad=True` — `.data` returns a view that is NOT a leaf with grad-tracking. (Alternative: `with t.no_grad(): p.add_(...)`.)

**Anatomy.**
```python
# Eval forward — no graph, no grads.
with t.no_grad():
    val_pred = model(x_val); val_loss = ((val_pred - y_val)**2).mean().item()
# Train batch — build a graph, backward, then in-place update via .data (or under no_grad).
pred = model(x_train); loss = ((pred - y_train)**2).mean()
loss.backward()                            # populates p.grad
p.data.add_(p.grad, alpha=-lr)             # Atom B: in-place leaf update via .data
```

**Why both atoms together.** Real ARENA training-vs-eval loops interleave them. If you do the inplace update WITHOUT `.data` and without `with t.no_grad():`, PyTorch throws. If you do the eval forward WITHOUT `t.no_grad()`, you build an autograd graph that you then never use.

### Composite Exercise — no-grad eval forward then in-place leaf update on the train batch

**Atoms exercised together**: `inference-mode-step`, `inplace-param-update`

Implement `cx16_eval_then_step(W, x_train, y_train, x_val, y_val, lr)`:

1. **Eval forward under `t.no_grad()`** — compute `val_pred = x_val @ W`, `val_loss = ((val_pred - y_val) ** 2).mean().item()`. Capture as `val_loss_v`.
2. **Train forward + backward** — compute `pred = x_train @ W`, `loss = ((pred - y_train) ** 2).mean()`, then `loss.backward()`.
3. **In-place leaf update**: `W.data.add_(W.grad, alpha=-lr)`. (Equivalently any in-place form on `W.data`. The `_` suffix and the `.data` view are BOTH necessary on a leaf with `requires_grad=True`.)

Return `(val_loss_v, W_after_data_clone)` where `W_after_data_clone = W.data.clone()`.

The test verifies: (a) the in-place op was on `.data` (W's `id` doesn't change between calls); (b) `W` is still the same Python object with `requires_grad=True` after the step; (c) the eval forward result matches a no_grad reference (so it really WAS the pre-step W); (d) the post-step `W` matches a manual reference `W_before - lr * W.grad`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx16_eval_then_step(W, x_train, y_train, x_val, y_val, lr):
    """No-grad eval then in-place SGD step via W.data.add_().

    Returns (val_loss: float, W_data_after: Tensor).
    """
    raise NotImplementedError

def _test_cx16():
    t.manual_seed(0)
    d, k = 5, 3
    W = t.randn(d, k, requires_grad=True)
    x_train = t.randn(12, d)
    y_train = t.randn(12, k)
    x_val = t.randn(8, d)
    y_val = t.randn(8, k)
    lr = 0.05

    W_id_before = id(W)
    W_storage_ptr_before = W.data.data_ptr()
    W_before = W.data.clone()

    val_loss_v, W_after = cx16_eval_then_step(W, x_train, y_train, x_val, y_val, lr)

    # Case A: eval result matches no_grad reference computed from pre-step W.
    with t.no_grad():
        ref_val = ((x_val @ W_before - y_val) ** 2).mean().item()
    assert abs(val_loss_v - ref_val) < 1e-5, (
        f'eval forward must use pre-step W; got {val_loss_v}, want {ref_val}'
    )

    # Case B: in-place — id(W) and storage pointer unchanged.
    assert id(W) == W_id_before, 'W object identity must not change (in-place via .data)'
    assert W.data.data_ptr() == W_storage_ptr_before, (
        'W.data storage was reallocated — step was NOT in-place'
    )

    # Case C: requires_grad still True; W is still a leaf.
    assert W.requires_grad is True, 'W must still require grad after the step'
    assert W.is_leaf is True, 'W must remain a leaf tensor'

    # Case D: post-step W matches manual reference.
    # Compute expected by hand: run the train backward on a frozen copy.
    W_ref = W_before.clone().requires_grad_(True)
    pred_ref = x_train @ W_ref
    loss_ref = ((pred_ref - y_train) ** 2).mean()
    loss_ref.backward()
    expected_W = W_before - lr * W_ref.grad
    assert t.allclose(W.data, expected_W, atol=1e-6), (
        f'post-step W mismatch; max err {(W.data - expected_W).abs().max().item()}'
    )
    assert t.allclose(W_after, expected_W, atol=1e-6), 'returned W_after must equal in-place result'

    # Case E: doing the SAME thing without .data on the leaf would raise — sanity-document.
    Wx = t.randn(3, requires_grad=True)
    Wx.grad = t.ones_like(Wx)
    raised = False
    try:
        Wx.add_(Wx.grad, alpha=-0.1)  # NO .data, NO no_grad — PyTorch refuses.
    except RuntimeError:
        raised = True
    assert raised, (
        'sanity: PyTorch should refuse in-place leaf update without .data or no_grad — '
        'if this passed, the explanation of why we use .data is now wrong'
    )
    _dd_passed.add('cx16')

_test_cx16()

<details><summary>Show solution — cx16</summary>

```python
def cx16_eval_then_step(W, x_train, y_train, x_val, y_val, lr):
    # Atom A (inference-mode-step): eval under no_grad — no graph, no grads.
    with t.no_grad():
        val_pred = x_val @ W
        val_loss_v = ((val_pred - y_val) ** 2).mean().item()
    # Train forward + backward (grad mode ON — outside the no_grad block).
    pred = x_train @ W
    loss = ((pred - y_train) ** 2).mean()
    loss.backward()
    # Atom B (inplace-param-update): in-place leaf write via .data.
    W.data.add_(W.grad, alpha=-lr)
    return val_loss_v, W.data.clone()
```

The `.data` access is the load-bearing trick: `W` itself is a leaf with `requires_grad=True`, and PyTorch's autograd refuses any in-place op on such a leaf (would corrupt the grad graph for any consumer that still references it). `W.data` returns the same underlying storage as a non-leaf view, so the in-place add is legal. The equivalent fully-explicit form is `with t.no_grad(): W.add_(W.grad, alpha=-lr)` — both work, `.data` is the ARENA convention.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx16'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx16',
        'subtopics': ["PyTorch: Inference mode step", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()